# Prédiction du Churn Orbitel & Optimisation de Décision Business

Ce notebook présente la modélisation prédictive du churn (départ des clients) pour l'entreprise **Orbitel**.

L'originalité de ce projet réside dans le fait que nous n'optimisons pas uniquement des métriques statistiques classiques (comme l'accuracy ou le F1-score), mais nous cherchons à **minimiser le coût financier global** pour l'entreprise.

## Problématique Métier
Chaque type d'erreur de prédiction possède un coût financier distinct :
- **Faux Négatif (FN)** : Le modèle prédit qu'un client restera fidèle alors qu'il va partir $\rightarrow$ **Coût = 200 €** (perte totale du client sans action de rétention).
- **Faux Positif (FP)** : Le modèle prédit qu'un client va partir alors qu'il est fidèle $\rightarrow$ **Coût = 50 €** (dépense marketing inutile pour une offre promotionnelle).

La fonction de coût métier s'écrit :
$$\text{Coût Métier} = 200 \times \text{FN} + 50 \times \text{FP}$$

Notre objectif est de minimiser cette fonction.

## 1. Importation des librairies et configuration

In [ ]:
# Installation des dépendances (utile sur Google Colab).
# Ces librairies sont déjà préinstallées sur Colab ; cette cellule garantit
# simplement des versions compatibles. Vous pouvez l'ignorer en local.
%pip install -q pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score,
)

sns.set_theme(style="whitegrid")

# ============================================================================
# Code source intégré (auto-suffisant pour Google Colab)
# Ce notebook ne dépend d'AUCUN fichier local : toutes les fonctions du projet
# (src/model.py, run_analysis.py, test_pipeline.py) sont définies ci-dessous.
# ============================================================================

# Colonnes considérées comme du bruit (50 scores comportementaux non informatifs
# qui font sur-apprendre le modèle vu la petite taille du jeu de données).
NOISE_KEYWORDS = ['behavior_score', 'behaviour_score']


def load_data(file_path: str) -> pd.DataFrame:
    """Charge un CSV de façon robuste en testant les séparateurs usuels (, et ;)."""
    df = pd.read_csv(file_path, sep=',')
    if df.shape[1] <= 1:
        df = pd.read_csv(file_path, sep=';')
    print(f"Chargement réussi : {file_path} (Taille : {df.shape[0]} lignes, {df.shape[1]} colonnes)")
    return df


def detect_columns(df: pd.DataFrame):
    """Détecte dynamiquement les colonnes de réclamations et de satisfaction."""
    cols = df.columns.tolist()

    reclam_keywords = ['reclamation', 'complaint', 'claim', 'nb_rec', 'num_rec', 'reclamations']
    reclam_col = None
    for kw in reclam_keywords:
        for col in cols:
            if kw in col.lower():
                reclam_col = col
                break
        if reclam_col:
            break

    satisf_keywords = ['satisfaction', 'satisf', 'score_sat', 'sat_level']
    satisf_col = None
    for kw in satisf_keywords:
        for col in cols:
            if kw in col.lower():
                satisf_col = col
                break
        if satisf_col:
            break

    if not reclam_col:
        reclam_col = 'reclamations'
        print(f"Attention : Colonne de réclamations non trouvée. Valeur par défaut : '{reclam_col}'")
    else:
        print(f"Colonne de réclamations détectée : '{reclam_col}'")

    if not satisf_col:
        satisf_col = 'satisfaction'
        print(f"Attention : Colonne de satisfaction non trouvée. Valeur par défaut : '{satisf_col}'")
    else:
        print(f"Colonne de satisfaction détectée : '{satisf_col}'")

    return reclam_col, satisf_col


def add_frustration_feature(df: pd.DataFrame) -> pd.DataFrame:
    """Crée la variable 'frustration' (0/1) : réclamations > 5 ET satisfaction < 2.5."""
    df_copy = df.copy()
    reclam_col, satisf_col = detect_columns(df_copy)

    if reclam_col in df_copy.columns and satisf_col in df_copy.columns:
        df_copy['frustration'] = np.where(
            (df_copy[reclam_col] > 5) & (df_copy[satisf_col] < 2.5), 1, 0
        )
        print("Variable 'frustration' créée avec succès.")
    else:
        df_copy['frustration'] = 0
        print("Attention : Colonnes requises absentes. Variable 'frustration' initialisée à 0.")

    return df_copy


def calculate_business_cost(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Coût métier = 200 * FN + 50 * FP."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    cost = 200 * fn + 50 * fp
    return {
        'total_cost': float(cost),
        'fn_cost': float(200 * fn),
        'fp_cost': float(50 * fp),
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
    }


def train_baseline_model(X_train, y_train) -> LogisticRegression:
    """Régression logistique de base (paramètres par défaut)."""
    model = LogisticRegression(random_state=42)
    model.fit(X_train, y_train)
    return model


def train_improved_model(X_train, y_train, class_weight='balanced') -> Pipeline:
    """Pipeline : StandardScaler + LogisticRegression (max_iter=1000, class_weight)."""
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=1000, class_weight=class_weight, random_state=42)),
    ])
    pipeline.fit(X_train, y_train)
    return pipeline


def evaluate_model(model, X, y_true, threshold: float = 0.5) -> dict:
    """Évalue le modèle pour un seuil de décision donné."""
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X)[:, 1]
        y_pred = (probs >= threshold).astype(int)
    else:
        y_pred = model.predict(X)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    cost_metrics = calculate_business_cost(
        y_true.values if hasattr(y_true, 'values') else y_true, y_pred
    )

    return {
        'threshold': threshold,
        'accuracy': acc, 'precision': prec, 'recall': rec, 'f1_score': f1,
        **cost_metrics,
    }


def optimize_threshold(model, X, y_true) -> tuple:
    """Cherche le seuil (0.1 à 0.9) qui minimise le coût métier total."""
    thresholds = np.linspace(0.1, 0.9, 9)
    results = [evaluate_model(model, X, y_true, threshold=t) for t in thresholds]
    df_results = pd.DataFrame(results)

    best_idx = df_results['total_cost'].idxmin()
    best_threshold = df_results.loc[best_idx, 'threshold']
    best_metrics = df_results.loc[best_idx].to_dict()
    return best_threshold, best_metrics, df_results


def prepare_features(df_train, df_test, df_new=None, drop_noise=True):
    """Sépare X/y, supprime identifiants ET features bruitées, garde le numérique, impute (médiane).

    drop_noise=True : retire les colonnes 'behavior_score_*' (50 variables non informatives
    qui provoquent du sur-apprentissage sur un petit jeu de données).
    """
    target_col = 'Churn_Y'
    if target_col not in df_train.columns:
        for col in df_train.columns:
            if col.lower() in ('churn_y', 'churn'):
                target_col = col
                break

    y_train = df_train[target_col] if target_col in df_train.columns else None
    y_test = df_test[target_col] if target_col in df_test.columns else None

    cols_to_drop = [target_col] if target_col in df_train.columns else []
    id_keywords = ['id', 'client', 'customer', 'uuid']
    for col in df_train.columns:
        if any(kw in col.lower() for kw in id_keywords) and col != target_col:
            cols_to_drop.append(col)

    # Suppression des features bruitées (scores comportementaux)
    if drop_noise:
        noise_cols = [c for c in df_train.columns if any(kw in c.lower() for kw in NOISE_KEYWORDS)]
        if noise_cols:
            print(f"Suppression de {len(noise_cols)} features bruitées (ex. '{noise_cols[0]}' ... '{noise_cols[-1]}').")
            cols_to_drop.extend(noise_cols)

    X_train = df_train.drop(columns=cols_to_drop, errors='ignore').select_dtypes(include=[np.number])
    X_test = df_test.drop(columns=cols_to_drop, errors='ignore').select_dtypes(include=[np.number])
    X_new = None
    if df_new is not None:
        X_new = df_new.drop(columns=cols_to_drop, errors='ignore').select_dtypes(include=[np.number])

    print(f"Features conservées ({X_train.shape[1]}) : {list(X_train.columns)}")

    imputer = SimpleImputer(strategy='median')
    X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
    X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)
    X_new_imp = None
    if X_new is not None:
        X_new_imp = pd.DataFrame(imputer.transform(X_new), columns=X_new.columns)

    return X_train_imp, y_train, X_test_imp, y_test, X_new_imp


def generate_synthetic_data(n_samples=1000, random_seed=42) -> pd.DataFrame:
    """Génère des données synthétiques réalistes pour Orbitel."""
    np.random.seed(random_seed)

    client_ids = [f"CUST_{i:05d}" for i in range(1, n_samples + 1)]
    age = np.random.randint(18, 80, size=n_samples)
    anciennete = np.random.randint(1, 120, size=n_samples)
    facture_mensuelle = np.random.uniform(20.0, 120.0, size=n_samples)

    reclamations = np.clip(np.random.negative_binomial(n=2, p=0.4, size=n_samples), 0, 12)
    satisfaction = np.clip(5.0 - (reclamations * 0.4) - np.random.uniform(0, 1.0, size=n_samples), 1.0, 5.0)

    logit = (
        -2.5
        + 0.5 * reclamations
        - 1.2 * (satisfaction - 3)
        + 0.02 * facture_mensuelle
        - 0.01 * age
    )
    prob_churn = 1 / (1 + np.exp(-logit))
    churn_y = np.random.binomial(1, prob_churn)

    return pd.DataFrame({
        'ClientID': client_ids,
        'Age': age,
        'Anciennete': anciennete,
        'FactureMensuelle': facture_mensuelle,
        'reclamations': reclamations,
        'satisfaction': satisfaction,
        'Churn_Y': churn_y,
    })


print("Librairies chargées et fonctions du projet définies avec succès.")

## 2. Chargement des données ou génération de données de simulation

Ce notebook est **auto-suffisant** : si les fichiers `churn_train.csv`, `churn_test.csv` et `churn_new.csv` ne sont pas présents, des données simulées sont générées automatiquement. Aucun fichier externe n'est donc requis pour l'exécuter sur **Google Colab**.

> **Utiliser vos vraies données sur Colab :** déposez vos fichiers via le menu *Fichiers* (icône dossier à gauche), ou exécutez `from google.colab import files; files.upload()` dans une cellule, puis relancez la cellule ci-dessous. Les fichiers doivent être nommés `churn_train.csv`, `churn_test.csv` et (optionnellement) `churn_new.csv`.

In [ ]:
train_path = 'churn_train.csv'
test_path = 'churn_test.csv'
new_path = 'churn_new.csv'

if not (os.path.exists(train_path) and os.path.exists(test_path)):
    print("Données réelles non détectées. Génération de données synthétiques pour démonstration...")

    df_train = generate_synthetic_data(n_samples=1000, random_seed=42)
    df_test = generate_synthetic_data(n_samples=300, random_seed=100)
    df_new = generate_synthetic_data(n_samples=150, random_seed=999).drop(columns=['Churn_Y'])

    df_train.to_csv(train_path, index=False, sep=';')
    df_test.to_csv(test_path, index=False, sep=';')
    df_new.to_csv(new_path, index=False, sep=';')
else:
    print("Données réelles détectées. Chargement des fichiers en cours...")
    df_train = load_data(train_path)
    df_test = load_data(test_path)
    df_new = load_data(new_path) if os.path.exists(new_path) else None

print(f"Jeu d'entraînement : {df_train.shape}")
print(f"Jeu de validation   : {df_test.shape}")

## 3. Partie 1 : Modèle Initial (Régression Logistique de Base)
Nous commençons par préparer les données brutes sans transformation et entraînons un modèle initial avec les paramètres par défaut.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

X_train, y_train, X_test, y_test, X_new = prepare_features(df_train, df_test, df_new)

print("Entraînement du modèle de base...")
# Le modèle de base utilise des données NON standardisées et max_iter par défaut (100).
# Cela provoque normalement un ConvergenceWarning : c'est exactement le problème que
# la Partie 2 corrige (standardisation + max_iter=1000). On masque ce warning attendu
# uniquement ici afin de garder une sortie lisible.
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    baseline_model = train_baseline_model(X_train, y_train)

# Évaluation au seuil par défaut de 0.5
baseline_metrics = evaluate_model(baseline_model, X_test, y_test, threshold=0.5)

print("(Note : le modèle de base ne converge pas totalement — c'est volontaire ; corrigé en Partie 2.)")
print(f"Coût Métier Initial (Seuil 0.5) : {baseline_metrics['total_cost']:.2f} €")
print(f"Structure des erreurs : FN = {baseline_metrics['FN']} ({baseline_metrics['fn_cost']:.0f} €), FP = {baseline_metrics['FP']} ({baseline_metrics['fp_cost']:.0f} €)")

## 4. Partie 2 : Amélioration du Modèle (Standardisation & Gestion du Déséquilibre)
Nous appliquons les améliorations techniques :
- Standardisation des données (`StandardScaler`) pour accélérer la convergence (et faire disparaître le warning de limite d'itérations).
- Équilibrage des classes (`class_weight='balanced'`) pour pénaliser les erreurs sur la classe minoritaire (le churn).

In [ ]:
print("Entraînement du modèle amélioré...")
improved_model = train_improved_model(X_train, y_train, class_weight='balanced')
improved_metrics = evaluate_model(improved_model, X_test, y_test, threshold=0.5)

print(f"Coût Métier Modèle Amélioré (Seuil 0.5) : {improved_metrics['total_cost']:.2f} €")
print(f"Structure des erreurs : FN = {improved_metrics['FN']} ({improved_metrics['fn_cost']:.0f} €), FP = {improved_metrics['FP']} ({improved_metrics['fp_cost']:.0f} €)")
print(f"Gain financier : {baseline_metrics['total_cost'] - improved_metrics['total_cost']:.2f} €")

## 5. Partie 3 : Variable 'Frustration' et Optimisation du Seuil
Nous intégrons la variable métier `'frustration'` et cherchons le seuil optimal entre 0.1 et 0.9 pour minimiser le coût financier final.

In [ ]:
# Création de la variable frustration
df_train_frust = add_frustration_feature(df_train)
df_test_frust = add_frustration_feature(df_test)
df_new_frust = add_frustration_feature(df_new) if df_new is not None else None

X_train_frust, y_train, X_test_frust, y_test, X_new_frust = prepare_features(df_train_frust, df_test_frust, df_new_frust)

# Entraînement du modèle final
final_model = train_improved_model(X_train_frust, y_train, class_weight='balanced')

# Recherche du seuil optimal
best_threshold, best_metrics, df_thresholds = optimize_threshold(final_model, X_test_frust, y_test)
df_thresholds

### Visualisation des résultats
Tracé du coût métier total en fonction du seuil de classification.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(df_thresholds['threshold'], df_thresholds['total_cost'], marker='o', linewidth=2.5, color='#e74c3c', label='Coût métier total')
plt.plot(df_thresholds['threshold'], df_thresholds['fn_cost'], '--', color='#2ecc71', label='Coût Faux Négatifs (Clients perdus)')
plt.plot(df_thresholds['threshold'], df_thresholds['fp_cost'], '--', color='#3498db', label='Coût Faux Positifs (Campagnes inutiles)')

# Indiquer le seuil optimal
plt.axvline(x=best_threshold, color='#8e44ad', linestyle=':', linewidth=2, label=f'Seuil Optimal ({best_threshold:.1f})')
plt.scatter(best_threshold, best_metrics['total_cost'], color='#8e44ad', s=150, zorder=5)

plt.title("Évolution du Coût Métier Global en fonction du Seuil de Décision", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Seuil de Décision (Classification Threshold)", fontsize=12)
plt.ylabel("Coût Financier pour l'entreprise (€)", fontsize=12)
plt.xticks(np.arange(0.1, 1.0, 0.1))
plt.legend(fontsize=11, loc='upper center')
plt.tight_layout()
plt.show()

## 6. Génération des prédictions finales
Nous appliquons notre meilleur modèle avec le seuil optimal sur les nouveaux clients.

In [ ]:
if X_new_frust is not None:
    probs_new = final_model.predict_proba(X_new_frust)[:, 1]
    preds_new = (probs_new >= best_threshold).astype(int)
    
    df_predictions = pd.DataFrame({
        'Probabilite_Churn': probs_new,
        'Pred_Churn_Y': preds_new
    })
    
    # Insérer l'ID s'il existe
    id_cols = [col for col in df_new.columns if any(kw in col.lower() for kw in ['id', 'client', 'customer'])]
    if id_cols:
        df_predictions.insert(0, id_cols[0], df_new[id_cols[0]])
        
    output_file = 'predictions_finales.csv'
    df_predictions.to_csv(output_file, index=False, sep=';')
    print(f"Prédictions enregistrées dans '{output_file}'")
    print(f"Nombre de churneurs détectés : {sum(preds_new)} sur {len(preds_new)} clients.")
else:
    print("Fichier churn_new.csv non disponible.")